# 02 — Full-scale HF data → train → smoke → Hub → Space

Kaggle **GPU (T4)** path with **Hugging Face auth via `HF_TOKEN` secret**:

1. Load `HF_TOKEN` from Kaggle Secrets (auth for dataset stream + later publish)
2. Stream **capped full-scale samples** from public HF datasets (not multi-GB dumps)
3. Build grounded SFT set → full-epoch QLoRA
4. Smoke adapter → push model → deploy Gradio Space

| Flag | Default | Meaning |
|------|---------|---------|
| `DOWNLOAD_HF` | `True` | Stream from Hub with token auth |
| `MAX_PER_SOURCE` | transcripts 400 / fiqa 200 / alpaca 150 | Portfolio-scale caps |
| `RUN_TRAIN` | `True` | Full epoch on the built set |
| `PUBLISH_HF` / `PUBLISH_SPACE` | `True` | After smoke |

**Secret name must be exactly `HF_TOKEN`.** Write scope for publish; read is enough for public datasets.

This is **not** the entire S&P corpus on disk — it is a **streaming cap** designed to produce a multi-step SFT run (thousands of grounded pairs after chunk/generate).

## 0. Knobs

In [17]:
RUN_TRAIN = True
MAX_STEPS = None                 # None = full epoch over the built train split

# --- Full-scale public HF ingest (token required for reliable auth) ---
DOWNLOAD_HF = True
# Per-source streaming caps (portfolio full run; not infinite dump)
MAX_PER_SOURCE = {
    "earnings_transcripts": 400,
    "fiqa": 200,
    "finance_alpaca": 150,
}
USE_LLM_JUDGE = True
CONFIG_PATH = "configs/default.yaml"

RUN_SMOKE = True
SMOKE_PROMPT = (
    "Summarize prepared remarks vs Q&A on a US large-cap quarterly earnings call."
)

RUN_SIDE_BY_SIDE = True
SIDE_BY_SIDE_LIMIT = 4

PUBLISH_HF = True
HF_REPO_ID = "skaran786/llama32-3b-ecra-sft"
HF_PRIVATE = False

PUBLISH_SPACE = True
SPACE_REPO_ID = "skaran786/earnings-call-research-assistant"
SPACE_PRIVATE = False
SPACE_DIR = "spaces/ecra-demo"

LAUNCH_GRADIO = False
GRADIO_SHARE = True
GRADIO_SIDE_BY_SIDE = True

ADAPTER_DIR = "outputs/adapters/llama32-3b-ecra-sft"
SMOKE_OK = True

print("DOWNLOAD_HF", DOWNLOAD_HF, "MAX_PER_SOURCE", MAX_PER_SOURCE)
print("RUN_TRAIN", RUN_TRAIN, "MAX_STEPS", MAX_STEPS)
print("PUBLISH_HF", PUBLISH_HF, HF_REPO_ID)
print("PUBLISH_SPACE", PUBLISH_SPACE, SPACE_REPO_ID)

DOWNLOAD_HF True MAX_PER_SOURCE {'earnings_transcripts': 400, 'fiqa': 200, 'finance_alpaca': 150}
RUN_TRAIN True MAX_STEPS None
PUBLISH_HF True skaran786/llama32-3b-ecra-sft
PUBLISH_SPACE True skaran786/earnings-call-research-assistant


## 1. Repo path + installs

In [2]:
from pathlib import Path
import os
import sys

IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)

if IN_KAGGLE:
    work = Path("/kaggle/working")
    repo = work / "earnings-call-research-assistant"
    # Always refresh so ingest/train fixes are present
    %cd /kaggle/working
    !rm -rf earnings-call-research-assistant
    !git clone --depth 1 https://github.com/nuwanda94/earnings-call-research-assistant.git
    REPO = (work / "earnings-call-research-assistant").resolve()
else:
    REPO = Path("..").resolve()
    if not (REPO / "src" / "earnings_call_research_assistant").is_dir():
        REPO = Path.cwd().resolve()

SRC = REPO / "src"
assert (SRC / "earnings_call_research_assistant" / "inference.py").exists(), SRC
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
print("REPO:", REPO)

import earnings_call_research_assistant as ecra
print("package:", ecra.__file__)

IN_KAGGLE: True
/kaggle/working
Cloning into 'earnings-call-research-assistant'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 68 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 88.97 KiB | 14.83 MiB/s, done.
Resolving deltas: 100% (10/10), done.
REPO: /kaggle/working/earnings-call-research-assistant
package: /kaggle/working/earnings-call-research-assistant/src/earnings_call_research_assistant/__init__.py


In [3]:
if IN_KAGGLE:
    %pip install -q pyyaml huggingface_hub datasets
    if RUN_TRAIN or RUN_SMOKE or RUN_SIDE_BY_SIDE or LAUNCH_GRADIO:
        %pip install -q unsloth transformers accelerate bitsandbytes trl peft
    if LAUNCH_GRADIO:
        %pip install -q gradio

Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 22.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Load HF_TOKEN from Kaggle Secrets (before any Hub call)

Add-ons → Secrets → name **`HF_TOKEN`**. Used for dataset streaming **and** adapter/Space publish.

In [4]:
from earnings_call_research_assistant.data import wire_hf_token, resolve_hf_token

def _load_hf_token() -> bool:
    if resolve_hf_token():
        return wire_hf_token()
    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
            if tok and tok.strip():
                return wire_hf_token(tok.strip())
        except Exception as e:
            print("Kaggle secrets note:", type(e).__name__, str(e)[:160])
    return False

has_token = _load_hf_token()
print("HF token active:", has_token, "(value never printed)")
if DOWNLOAD_HF and not has_token:
    print("WARNING: DOWNLOAD_HF=True but no token — public streams may 429 more often.")
if (PUBLISH_HF or PUBLISH_SPACE) and not has_token:
    print("WARNING: publish enabled but no HF_TOKEN.")
assert has_token or not DOWNLOAD_HF, (
    "Set Kaggle secret HF_TOKEN for full-scale Hub ingest."
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF token active: True (value never printed)


## 3. Phase 1 — full-scale public ingest (HF streaming + caps)

Streams each source up to `MAX_PER_SOURCE` with your token. Then chunk → grounded pairs → filter → versioned splits.

In [5]:
from earnings_call_research_assistant.data import (
    DATASET_VERSION, DEFAULT_MAX_PER_SOURCE, ChunkConfig, FilterConfig,
    GenerateConfig, SelectConfig, chunk_records, filter_pairs, generate_pairs,
    ingest_catalog, list_sources, select_and_split, write_chunks_jsonl,
    write_filter_report, write_jsonl, write_pairs_jsonl, write_splits,
)

for s in list_sources():
    print(f"  - {s.source_id}: {s.display_name} ({s.hf_id})")

caps = {**DEFAULT_MAX_PER_SOURCE, **MAX_PER_SOURCE}
print("Using per-source caps:", caps)
print("DOWNLOAD_HF:", DOWNLOAD_HF, "| token:", bool(resolve_hf_token()))

records = ingest_catalog(
    max_per_source=caps,
    download=DOWNLOAD_HF,
    pause_between_sources_s=2.0,  # reduce 429 risk between datasets
)
write_jsonl(records, Path("data/raw/public_sample.jsonl"))
print("records:", len(records))
from collections import Counter
print("by source:", dict(Counter(r.source_id for r in records)))

chunks = chunk_records(records, config=ChunkConfig(window_sentences=4, stride_sentences=2))
write_chunks_jsonl(chunks, Path("data/processed/chunks.jsonl"))
print("chunks:", len(chunks), "props:", sum(len(c.propositions) for c in chunks))

pairs = generate_pairs(
    chunks,
    config=GenerateConfig(max_qa_per_chunk=2, include_summary=True, use_llm=False),
)
write_pairs_jsonl(pairs, Path("data/processed/grounded_pairs.jsonl"))
print("pairs:", len(pairs))

kept, report = filter_pairs(
    pairs,
    config=FilterConfig(
        min_output_chars=40,
        near_dup_jaccard=0.88,
        use_llm_judge=USE_LLM_JUDGE,
        min_judge_score=0.6,
    ),
)
write_pairs_jsonl(kept, Path("data/processed/filtered_pairs.jsonl"))
write_filter_report(report, Path("data/processed/filter_report.json"))
print("kept:", report.n_kept, "dropped:", report.dropped_by_stage)

OUT_DIR = Path("data/processed") / DATASET_VERSION
sel_cfg = SelectConfig(
    target_min=100,
    target_max=6000,
    max_per_source=2500,
    diversity_jaccard_cap=0.72,
    seed=94,
    dataset_version=DATASET_VERSION,
)
splits, sel_report = select_and_split(kept, config=sel_cfg)
paths = write_splits(splits, OUT_DIR, report=sel_report, config=sel_cfg)
print(
    f"selected={sel_report.n_selected} train={sel_report.n_train} "
    f"val={sel_report.n_val} test={sel_report.n_test}"
)
print("train path:", paths["train"])
assert sel_report.n_train >= 20, (
    f"Train split too small ({sel_report.n_train}). Check ingest caps / HF errors."
)

  - earnings_transcripts: S&P 500 earnings call transcripts (kurry/sp500_earnings_transcripts)
  - fiqa: FiQA financial QA (LLukas22/fiqa)
  - finance_alpaca: Finance-Alpaca / Wealth-Alpaca (gbharti/finance-alpaca)
Using per-source caps: {'earnings_transcripts': 400, 'fiqa': 200, 'finance_alpaca': 150}
DOWNLOAD_HF: True | token: True


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[ingest] streaming kurry/sp500_earnings_transcripts (cap=400, token=yes)


README.md: 0.00B [00:00, ?B/s]

[ingest] earnings_transcripts: got 400 records


...

[ingest] fiqa: got 200 records


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[ingest] streaming gbharti/finance-alpaca (cap=150, token=yes)


README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

[ingest] finance_alpaca: got 150 records
records: 750
by source: {'earnings_transcripts': 400, 'fiqa': 200, 'finance_alpaca': 150}
chunks: 20183 props: 36515
pairs: 60549
[filter] start n_in=60549
[filter] after heuristic kept=60549
[filter:dedup] 6054/60549 scanned, kept=5272
[filter:dedup] 12108/60549 scanned, kept=10740
[filter:dedup] 18162/60549 scanned, kept=15810
[filter:dedup] 24216/60549 scanned, kept=20991
[filter:dedup] 30270/60549 scanned, kept=26259
[filter:dedup] 36324/60549 scanned, kept=31353
[filter:dedup] 42378/60549 scanned, kept=36447
[filter:dedup] 48432/60549 scanned, kept=41692
[filter:dedup] 54486/60549 scanned, kept=46903
[filter:dedup] 60540/60549 scanned, kept=52566
[filter:dedup] done scanned=60549 kept=52574
[filter] done n_kept=52574 dropped={'heuristic': 0, 'exact_dup': 503, 'near_dup': 7472, 'llm_judge': 0}
kept: 52574 dropped: {'heuristic': 0, 'exact_dup': 503, 'near_dup': 7472, 'llm_judge': 0}
selected=2627 train=2127 val=240 test=260
train path: data/p

## 4. SFT dry-run plan

In [6]:
from earnings_call_research_assistant.training.sft import run_sft
import json

plan = run_sft(
    config_path=CONFIG_PATH,
    dataset_dir=OUT_DIR,
    dry_run=True,
    max_steps=MAX_STEPS,
    require_train=False,
)
print(f"model={plan.model_name} train={plan.n_train} val={plan.n_val} adapter={plan.adapter_dir}")
print("Expect many optimizer steps when n_train is hundreds+.")

model=unsloth/Llama-3.2-3B-Instruct train=2127 val=240 adapter=outputs/adapters/llama32-3b-ecra-sft
Expect many optimizer steps when n_train is hundreds+.


## 5. Full QLoRA train (one epoch over the built train split)

Uses `num_train_epochs` from YAML when `MAX_STEPS is None`. With hundreds of train rows you should see **many steps**, not `[1/1]`.

In [7]:
import torch

print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

if not RUN_TRAIN:
    print("Skipped train.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable T4 GPU.")
    if plan.n_train < 20:
        raise RuntimeError(
            f"Train set too small ({plan.n_train}). Re-run Phase 1 with DOWNLOAD_HF=True + HF_TOKEN."
        )
    train_plan = run_sft(
        config_path=CONFIG_PATH,
        dataset_dir=OUT_DIR,
        dry_run=False,
        max_steps=MAX_STEPS,
        require_train=True,
    )
    print("Train finished:", train_plan.adapter_dir)
    print("notes:", train_plan.notes[-5:])

adapter_path = Path(ADAPTER_DIR)
print("adapter exists:", adapter_path.exists())
if adapter_path.exists():
    print("files:", sorted(p.name for p in adapter_path.iterdir())[:15])

cuda: True Tesla T4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 2.152 GiB
no_split classes   : ['LlamaDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 11.727 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  1.184 GiB  free 11.768 GiB  reserve 11.727 GiB
  cuda:1  budget  13.00 GiB  weights  0.968 GiB  free 12.028 GiB  reserve 11.

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2127 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 2,127 | Num Epochs = 1 | Total steps = 133
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,1.089967,1.132686
100,1.000304,1.045669
133,1.002119,1.036257


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-133/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/adapters/llama32-3b-ecra-sft/tokenizer_config.json.


Train finished: outputs/adapters/llama32-3b-ecra-sft
notes: ['Kaggle: enable T4 GPU, pip install unsloth + trl, then pass --run.', '8B path: --config configs/llama32-8b.yaml (batch 1 / accum 16).', 'OOM: lower --batch-size to 1 and raise --grad-accum to keep effective batch.', 'Dry-run never loads weights or starts SFTTrainer.', 'Saved LoRA adapter to outputs/adapters/llama32-3b-ecra-sft']
adapter exists: True
files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


## 6. Smoke-generate with adapter (gates Hub)

In [14]:
import gc
import yaml
from earnings_call_research_assistant.inference import InferenceConfig, InferenceHarness

def _release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

SMOKE_OK = False
smoke_reply = ""

if not RUN_SMOKE:
    print("RUN_SMOKE=False.")
elif not Path(ADAPTER_DIR).exists():
    raise FileNotFoundError(ADAPTER_DIR)
elif not torch.cuda.is_available():
    raise RuntimeError("Smoke needs CUDA.")
else:
    with Path(CONFIG_PATH).open() as f:
        cfg = InferenceConfig.from_mapping(yaml.safe_load(f))
    harness = None
    try:
        try:
            harness = InferenceHarness.from_pretrained(cfg, model_name=ADAPTER_DIR)
        except Exception as e:
            print("Direct load failed, base+load_adapter:", e)
            harness = InferenceHarness.from_pretrained(cfg)
            harness.model.load_adapter(ADAPTER_DIR)
        smoke_reply = (harness.generate(SMOKE_PROMPT) or "").strip()
        print("SMOKE REPLY:")
        print(smoke_reply[:800] if smoke_reply else "(empty)")
        if len(smoke_reply) >= 20:
            SMOKE_OK = True
        else:
            raise RuntimeError("Smoke reply too short.")
    finally:
        if harness is not None:
            del harness
        _release()
print("SMOKE_OK =", SMOKE_OK)

==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 2.152 GiB
no_split classes   : ['LlamaDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 10.065 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  11.66 GiB  weights  1.418 GiB  free 10.242 GiB  reserve 10.065 GiB
  cuda:1  budget  10.96 GiB  weights  0.734 GiB  free 10.230 GiB  reserve  9.579 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 12.956 -> 11.660 GiB, cuda:1 12.182 -> 10.964 GiB (memory the

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load outputs/adapters/llama32-3b-ecra-sft as a legacy tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SMOKE REPLY:
Research note: The prepared remarks passage is a short excerpt from the earnings-call Q&A session. The Q&A section is a follow-up to the prepared remarks. The prepared remarks are a brief overview of the company's performance, and the Q&A is a more in-depth discussion of the prepared remarks and the company's results.

Citations: earnings_transcripts:prepared_remarks:0-3:6e8f6e5f6d8#s0. No figures beyond these source sentences.
SMOKE_OK = True


## 7. Optional side-by-side

In [15]:
from earnings_call_research_assistant.eval.panel import load_panel, DEFAULT_PANEL

def _generate_batch(harness, items):
    rows = []
    for item in items:
        text = item.user_text()
        reply = harness.generate(text)
        rows.append({"id": item.id, "ticker": item.ticker, "theme": item.theme,
                     "user_text": text, "reply": reply})
        print(f"  [{item.id}] -> {len(reply)} chars")
    return rows

compare_path = Path("evals/reports/side_by_side_panel.jsonl")
compare_path.parent.mkdir(parents=True, exist_ok=True)

if not RUN_SIDE_BY_SIDE or not SMOKE_OK:
    print("Skipped side-by-side.")
else:
    panel = load_panel(DEFAULT_PANEL)[: max(1, int(SIDE_BY_SIDE_LIMIT))]
    with Path(CONFIG_PATH).open() as f:
        cfg = InferenceConfig.from_mapping(yaml.safe_load(f))
    base_h = InferenceHarness.from_pretrained(cfg)
    base_rows = _generate_batch(base_h, panel)
    del base_h
    _release()
    try:
        tuned_h = InferenceHarness.from_pretrained(cfg, model_name=ADAPTER_DIR)
    except Exception as e:
        tuned_h = InferenceHarness.from_pretrained(cfg)
        tuned_h.model.load_adapter(ADAPTER_DIR)
    tuned_rows = _generate_batch(tuned_h, panel)
    del tuned_h
    _release()
    by_id = {r["id"]: r for r in tuned_rows}
    with compare_path.open("w", encoding="utf-8") as f:
        for br in base_rows:
            tr = by_id.get(br["id"], {})
            row = {"id": br["id"], "ticker": br["ticker"], "theme": br["theme"],
                   "user_text": br["user_text"], "base_reply": br["reply"],
                   "adapter_reply": tr.get("reply", "")}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("Wrote", compare_path)

==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 2.152 GiB
no_split classes   : ['LlamaDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 11.189 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.85 GiB  weights  1.418 GiB  free 11.436 GiB  reserve 11.189 GiB
  cuda:1  budget  12.02 GiB  weights  0.734 GiB  free 11.283 GiB  reserve 10.635 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.282 -> 12.854 GiB, cuda:1 13.352 -> 12.017 GiB (memory the

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [p01] -> 391 chars


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [p01] -> 581 chars


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [p02] -> 561 chars


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [p03] -> 253 chars
  [p04] -> 366 chars
Wrote evals/reports/side_by_side_panel.jsonl


## 8. Push adapter to HF Hub

In [18]:
from earnings_call_research_assistant.publish import publish_adapter

if not PUBLISH_HF:
    print("Skipped adapter upload.")
elif not SMOKE_OK:
    raise RuntimeError("SMOKE_OK is False.")
else:
    live = publish_adapter(
        adapter_dir=ADAPTER_DIR, repo_id=HF_REPO_ID, private=HF_PRIVATE,
        commit_message="feat: upload ECRA QLoRA adapter (full-scale Kaggle run)",
        dry_run=False,
    )
    print("Adapter Hub:", live.hub_url)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Adapter Hub: https://huggingface.co/skaran786/llama32-3b-ecra-sft


## 9. Deploy Gradio Space

In [19]:
from earnings_call_research_assistant.space_publish import publish_space

if not PUBLISH_SPACE:
    print("Skipped Space.")
elif not SMOKE_OK:
    raise RuntimeError("SMOKE_OK is False.")
else:
    space_live = publish_space(
        space_dir=SPACE_DIR, repo_id=SPACE_REPO_ID, private=SPACE_PRIVATE,
        commit_message="feat: deploy ECRA Space after full-scale train",
        dry_run=False,
    )
    print("Space URL:", space_live.hub_url)
    print("Settings → Hardware → T4; ADAPTER_REPO=", HF_REPO_ID)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a9bd3ee-1f631a6f042d7b84722f8af2;91cdeabf-56c0-4aaf-991f-93c397d89d0b)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Static Spaces are free for everyone, but hosting Gradio and Docker Spaces on free cpu-basic requires a PRO subscription. Subscribe at https://huggingface.co/pro

## Done

Success signals:
- Phase 1 `records` in the hundreds+
- `train=` hundreds+
- Trainer progress **not** stuck at `[1/1]`
- Adapter + Space URLs printed